# Scientific context & Dataset downloading

# Modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# File name

In [ ]:
df_filename = "CERES_MODIS_TERRA_AQUA_2002_2020_Clement_et_al.csv"

# Read File

In [ ]:
def NaN_counter(pd_dataframe):
    # Count NaNs and percentage per column
    nan_counts = pd_dataframe.isna().sum()
    nan_percentage = (nan_counts / len(pd_dataframe)) * 100
    
    # Header
    print(f"{'Column':<30} {'NaNs':>10} {'% NaNs':>10}")
    print("-" * 52)
    
    # Rows
    for column in pd_dataframe.columns:
        print(f"{column:<30} {nan_counts[column]:>10} {nan_percentage[column]:>9.2f}%")

In [ ]:
df = pd.read_csv(df_filename)
print("Data loaded")

In [ ]:
NaN_counter(df)

In [ ]:
# lwp (liquid water path) from g/m2 to kg/m2
df['lwp'] = df['lwp'] * 1e-3

In [ ]:
# Create a new column for [ log(aerosol optical depth) ] and insert it right after 'od_aer'
if not 'log_od_aer' in df.columns:
    df.insert(
        df.columns.get_loc('od_aer') + 1,
        'log_od_aer',
        np.log(df['od_aer'].where(df['od_aer'] > 0, np.nan))
    )

In [ ]:
df.columns

In [ ]:
df

In [ ]:
df.describe()

### function to compute of trends

In [ ]:
# ----------------------------------------------------------
# Function to compute trend in a given feature
# ----------------------------------------------------------
def compute_trend(df_region, feature, first_year, last_year, saturated=False):
    """
    Computes the trend (slope) of a given feature over time for each lat/lon grid point.

    Parameters:
    - df_region: DataFrame containing columns 'lat', 'lon', 'year', 'month', and the feature to analyze.
    - feature: Name of the column in df_region for which the trend is computed.
    - first_year: First year to include in the trend calculation.
    - last_year: Last year to include in the trend calculation.
    - saturated: If True, saturates the slope to +1 or -1.

    Returns:
    - DataFrame with columns 'lat', 'lon', and the computed trend for the feature.
    """
    slopes = []

    for (lat, lon), g in df_region.groupby(["lat", "lon"]):
        # Sort the group by year and month
        g = g.sort_values(["year", "month"])

        # Restrict data to the specified years
        g = g[(g["year"] >= first_year) & (g["year"] <= last_year)]

        # Create a fractional time axis (year + fractional month)
        time = (g["year"] + (g["month"] - 0.5) / 12).values
        y = g[feature].values

        # Remove NaN values from both time and feature arrays
        mask = ~np.isnan(time) & ~np.isnan(y)
        t = time[mask]
        f = y[mask]

        # Compute the slope (trend) using linear regression
        if len(t) > 1:
            slope = np.polyfit(t, f, 1)[0]
        else:
            slope = np.nan

        # Saturate the slope if required
        if saturated:
            if slope > 0:
                slope = 1
            else:
                slope = -1

        slopes.append((lat, lon, slope))

    # Return a DataFrame with the computed slopes
    return pd.DataFrame(slopes, columns=["lat", "lon", feature+"_trend"])

# Plot tests

## Function to plot

In [ ]:
def plot_map(df_plot, variable_to_plot, title_plot):
    """
    Plots a global map with the provided data.

    Parameters:
    - df_plot: Pandas DataFrame containing the columns 'lat', 'lon', and the variable to plot.
    - variable_to_plot: Name of the DataFrame column to plot.
    - title_plot: Title of the plot.
    """
    # Extract unique coordinates
    lat_full = np.sort(df_plot["lat"].unique())
    lon_full = np.sort(df_plot["lon"].unique())

    # Create a grid of coordinates
    lon_grid, lat_grid = np.meshgrid(lon_full, lat_full)

    # Pivot the data onto the grid
    pivot = df_plot.set_index(["lat", "lon"])[variable_to_plot].reindex(
        pd.MultiIndex.from_product([lat_full, lon_full], names=["lat", "lon"])
    ).values.reshape(len(lat_full), len(lon_full))

    # Create the figure and axis with a global projection
    fig, ax = plt.subplots(
        subplot_kw={'projection': ccrs.PlateCarree()},
        figsize=(12, 6)
    )

    # Add country borders and coastlines
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='black')
    ax.coastlines()

    # Add grid lines
    gl = ax.gridlines(draw_labels=True)
    gl.xlabel_style = {'size': 12}
    gl.ylabel_style = {'size': 12}
    
    # Choose the colorbar color and define its boundaries
    if "land_sea_mask" in variable_to_plot:
        cmap='RdBu'
        vmin=0
        vmax=100
    elif (np.nanmin(pivot) * np.nanmax(pivot) < 0 or "trend" in variable_to_plot) and (not "od_aer" in variable_to_plot):
        cmap='RdBu_r'
        # we saturate the colorbar for low and high values
        max_abs = np.nanpercentile(np.abs(pivot), 95)
        vmin= -max_abs
        vmax= max_abs
    else:
        cmap='plasma'
        # we saturate the colorbar for low and high values
        vmin=np.nanpercentile(pivot, 5)
        vmax=np.nanpercentile(pivot, 95)

    # Plot the data
    pc = ax.pcolormesh(
        lon_grid, lat_grid, pivot,
        cmap=cmap,
        shading='auto',
        vmin=vmin,
        vmax=vmax,
        transform=ccrs.PlateCarree()
    )

    # Add a color bar
    cbar = plt.colorbar(pc, ax=ax, orientation='vertical', pad=0.05)
    cbar.ax.tick_params(labelsize=12)
    if "trend" in variable_to_plot:
        cbar.set_label(rf"{variable_to_plot} (/year)", fontsize=14)
    else:
        cbar.set_label(rf"{variable_to_plot}", fontsize=14)

    # Add a title
    plt.title(title_plot, fontsize=16)

    plt.tight_layout()
    plt.show()

## Values at a given month and given year

In [ ]:
variable_to_plot = "surface_temperature"
#variable_to_plot = "surface_albedo"
#variable_to_plot = "clt"
#variable_to_plot = "od_aer"
#variable_to_plot = "log_od_aer"
#variable_to_plot = "land_sea_mask"

year_to_plot = 2020
month_to_plot = 1      # 1 (January) to 12 (December)
title_plot = rf"{variable_to_plot} - year:{year_to_plot}, month:{month_to_plot}"

In [ ]:
df_plot = df[(df['year'] == year_to_plot) & (df['month'] == month_to_plot)]

In [ ]:
plot_map(df_plot, variable_to_plot, title_plot)

## Trends over a period (first year - last year)

### in surface temperature

In [ ]:
variable_to_plot_trend = "surface_temperature"
first_year, last_year = 2002, 2020
df_plot = compute_trend(df, variable_to_plot_trend, first_year, last_year, saturated=False)
variable_to_plot = variable_to_plot_trend + "_trend"
title_plot = rf"{variable_to_plot} - {first_year}-{last_year}"

In [ ]:
plot_map(df_plot, variable_to_plot, title_plot)

### in aerosol optical depth

In [ ]:
variable_to_plot_trend = "od_aer"
first_year, last_year = 2002, 2020
df_plot = compute_trend(df, variable_to_plot_trend, first_year, last_year, saturated=False)
variable_to_plot = variable_to_plot_trend + "_trend"
title_plot = rf"{variable_to_plot} - {first_year}-{last_year}"

In [ ]:
plot_map(df_plot, variable_to_plot, title_plot)

### in Top-Of-Atmosphere (TOA) albedo

In [ ]:
variable_to_plot_trend = "albedo_TOA"
first_year, last_year = 2002, 2020
df_plot = compute_trend(df, variable_to_plot_trend, first_year, last_year, saturated=False)
variable_to_plot = variable_to_plot_trend + "_trend"
title_plot = rf"{variable_to_plot} - {first_year}-{last_year}"

In [ ]:
plot_map(df_plot, variable_to_plot, title_plot)

### in cloud fraction

In [ ]:
variable_to_plot_trend = "clt"
first_year, last_year = 2002, 2020
df_plot = compute_trend(df, variable_to_plot_trend, first_year, last_year, saturated=False)
variable_to_plot = variable_to_plot_trend + "_trend"
title_plot = rf"{variable_to_plot} - {first_year}-{last_year}"

In [ ]:
plot_map(df_plot, variable_to_plot, title_plot)

### in droplet number

In [ ]:
variable_to_plot_trend = "scdnc"
first_year, last_year = 2002, 2020
df_plot = compute_trend(df, variable_to_plot_trend, first_year, last_year, saturated=False)
variable_to_plot = variable_to_plot_trend + "_trend"
title_plot = rf"{variable_to_plot} - {first_year}-{last_year}"

In [ ]:
plot_map(df_plot, variable_to_plot, title_plot)

### in surface albedo

In [ ]:
variable_to_plot_trend = "surface_albedo"
first_year, last_year = 2002, 2020
df_plot = compute_trend(df, variable_to_plot_trend, first_year, last_year, saturated=False)
variable_to_plot = variable_to_plot_trend + "_trend"
title_plot = rf"{variable_to_plot} - {first_year}-{last_year}"

In [ ]:
plot_map(df_plot, variable_to_plot, title_plot)

### in clear-sky albedo

In [ ]:
variable_to_plot_trend = "albedo_TOA_clear_sky"
first_year, last_year = 2002, 2020
df_plot = compute_trend(df, variable_to_plot_trend, first_year, last_year, saturated=False)
variable_to_plot = variable_to_plot_trend + "_trend"
title_plot = rf"{variable_to_plot} - {first_year}-{last_year}"

In [ ]:
plot_map(df_plot, variable_to_plot, title_plot)

# Define region

In [ ]:
min_lat, max_lat = -90.0, 90.0
min_lon, max_lon = -180., 180.

## Regions of interest

In [ ]:
region = 'Europe'

In [ ]:
region = 'Eastern North America'

In [ ]:
region = 'Northeastern Asia'

In [ ]:
region = 'India'

## region boundaries

In [ ]:
if region == 'Europe':    
    min_lat, max_lat = 30.0, 70.0
    min_lon, max_lon = -10., 75.0

if region == 'Eastern North America':
    min_lat, max_lat = 20., 60.
    min_lon, max_lon = -105., -30.

if region == 'Northeastern Asia':
    min_lat, max_lat = 15.0, 65.0
    min_lon, max_lon = 80.0, 160.0
    
if region == 'India':  
    min_lat, max_lat = 5., 30.
    min_lon, max_lon = 60.0, 105.0